In [1]:
import kagglehub
import os

# Download dataset
path = kagglehub.dataset_download("paultimothymooney/chest-xray-pneumonia")
print("Dataset Path:", path)

# Set paths
base_path = os.path.join(path, "chest_xray")

train_dir = os.path.join(base_path, "train")
val_dir   = os.path.join(base_path, "val")
test_dir  = os.path.join(base_path, "test")

c:\Users\gurra\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


100%|██████████| 2.29G/2.29G [12:26<00:00, 3.30MB/s]

Extracting files...


Dataset Path: C:\Users\gurra\.cache\kagglehub\datasets\paultimothymooney\chest-xray-pneumonia\versions\2


In [2]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator

IMG_SIZE = (224, 224)
BATCH_SIZE = 32

train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    zoom_range=0.2,
    horizontal_flip=True,
    shear_range=0.1
)

test_datagen = ImageDataGenerator(rescale=1./255)

train_data = train_datagen.flow_from_directory(
    train_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary'
)

val_data = test_datagen.flow_from_directory(
    val_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary'
)

test_data = test_datagen.flow_from_directory(
    test_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    shuffle=False
)

Found 5216 images belonging to 2 classes.
Found 16 images belonging to 2 classes.
Found 624 images belonging to 2 classes.


In [3]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Dropout, Flatten, Dense

cnn_model = Sequential([
    Conv2D(32, (3,3), activation='relu', input_shape=(224,224,3)),
    MaxPooling2D(2,2),

    Conv2D(64, (3,3), activation='relu'),
    MaxPooling2D(2,2),

    Conv2D(128, (3,3), activation='relu'),
    MaxPooling2D(2,2),

    Flatten(),

    Dense(128, activation='relu'),
    Dropout(0.5),

    Dense(1, activation='sigmoid')
])

cnn_model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

cnn_model.summary()

c:\Users\gurra\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 222, 222, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 109, 109, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 54, 54, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 52, 52, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 26, 26, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 86528)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │    11,075,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 11,169,089 (42.61 MB)

 Trainable params: 11,169,089 (42.61 MB)

 Non-trainable params: 0 (0.00 B)

In [4]:
cnn_history = cnn_model.fit(
    train_data,
    validation_data=val_data,
    epochs=10
)

Epoch 1/10
163/163 ━━━━━━━━━━━━━━━━━━━━ 347s 2s/step - accuracy: 0.8250 - loss: 0.4082 - val_accuracy: 0.8125 - val_loss: 0.4917
Epoch 2/10
163/163 ━━━━━━━━━━━━━━━━━━━━ 176s 1s/step - accuracy: 0.8867 - loss: 0.2874 - val_accuracy: 0.8750 - val_loss: 0.4955
Epoch 3/10
163/163 ━━━━━━━━━━━━━━━━━━━━ 170s 1s/step - accuracy: 0.8976 - loss: 0.2543 - val_accuracy: 0.8750 - val_loss: 0.5046
Epoch 4/10
163/163 ━━━━━━━━━━━━━━━━━━━━ 169s 1s/step - accuracy: 0.9013 - loss: 0.2343 - val_accuracy: 0.6250 - val_loss: 0.7842
Epoch 5/10
163/163 ━━━━━━━━━━━━━━━━━━━━ 166s 1s/step - accuracy: 0.9149 - loss: 0.2184 - val_accuracy: 0.6875 - val_loss: 0.6225
Epoch 6/10
163/163 ━━━━━━━━━━━━━━━━━━━━ 175s 1s/step - accuracy: 0.9231 - loss: 0.1931 - val_accuracy: 0.6250 - val_loss: 0.6188
Epoch 7/10
163/163 ━━━━━━━━━━━━━━━━━━━━ 170s 1s/step - accuracy: 0.9310 - loss: 0.1764 - val_accuracy: 0.6250 - val_loss: 0.7001
Epoch 8/10
163/163 ━━━━━━━━━━━━━━━━━━━━ 201s 1s/step - accuracy: 0.9327 - loss: 0.1775 - val_accu

In [5]:
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import GlobalAveragePooling2D

base_model = MobileNetV2(
    weights='imagenet',
    include_top=False,
    input_shape=(224,224,3)
)

base_model.trainable = False

tl_model = Sequential([
    base_model,
    GlobalAveragePooling2D(),
    Dense(128, activation='relu'),
    Dropout(0.5),
    Dense(1, activation='sigmoid')
])

tl_model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

tl_model.summary()

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 6s 1us/step


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │       163,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,422,081 (9.24 MB)

 Trainable params: 164,097 (641.00 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [6]:
tl_history = tl_model.fit(
    train_data,
    validation_data=val_data,
    epochs=10
)

Epoch 1/10
163/163 ━━━━━━━━━━━━━━━━━━━━ 215s 1s/step - accuracy: 0.8924 - loss: 0.2447 - val_accuracy: 0.6250 - val_loss: 0.8012
Epoch 2/10
163/163 ━━━━━━━━━━━━━━━━━━━━ 273s 1s/step - accuracy: 0.9402 - loss: 0.1589 - val_accuracy: 0.6875 - val_loss: 0.6685
Epoch 3/10
163/163 ━━━━━━━━━━━━━━━━━━━━ 174s 1s/step - accuracy: 0.9410 - loss: 0.1472 - val_accuracy: 0.7500 - val_loss: 0.3581
Epoch 4/10
163/163 ━━━━━━━━━━━━━━━━━━━━ 716s 4s/step - accuracy: 0.9480 - loss: 0.1321 - val_accuracy: 0.8125 - val_loss: 0.3027
Epoch 5/10
163/163 ━━━━━━━━━━━━━━━━━━━━ 313s 2s/step - accuracy: 0.9521 - loss: 0.1240 - val_accuracy: 0.7500 - val_loss: 0.3744
Epoch 6/10
163/163 ━━━━━━━━━━━━━━━━━━━━ 191s 1s/step - accuracy: 0.9503 - loss: 0.1227 - val_accuracy: 0.8125 - val_loss: 0.3904
Epoch 7/10
163/163 ━━━━━━━━━━━━━━━━━━━━ 244s 1s/step - accuracy: 0.9515 - loss: 0.1169 - val_accuracy: 0.8125 - val_loss: 0.4051
Epoch 8/10
163/163 ━━━━━━━━━━━━━━━━━━━━ 246s 2s/step - accuracy: 0.9563 - loss: 0.1165 - val_accu

In [7]:
cnn_loss, cnn_acc = cnn_model.evaluate(test_data)
tl_loss, tl_acc = tl_model.evaluate(test_data)

print("CNN Accuracy:", cnn_acc)
print("Transfer Learning Accuracy:", tl_acc)

20/20 ━━━━━━━━━━━━━━━━━━━━ 18s 875ms/step - accuracy: 0.8766 - loss: 0.4128
20/20 ━━━━━━━━━━━━━━━━━━━━ 13s 594ms/step - accuracy: 0.8830 - loss: 0.2821
CNN Accuracy: 0.8766025900840759
Transfer Learning Accuracy: 0.8830128312110901


In [8]:
import numpy as np
from sklearn.metrics import confusion_matrix, classification_report

y_true = test_data.classes

y_pred_cnn = (cnn_model.predict(test_data) > 0.5).astype(int)
y_pred_tl = (tl_model.predict(test_data) > 0.5).astype(int)

print("CNN Report:")
print(classification_report(y_true, y_pred_cnn))

print("Transfer Learning Report:")
print(classification_report(y_true, y_pred_tl))

20/20 ━━━━━━━━━━━━━━━━━━━━ 8s 355ms/step
20/20 ━━━━━━━━━━━━━━━━━━━━ 15s 657ms/step
CNN Report:
              precision    recall  f1-score   support

           0       0.96      0.70      0.81       234
           1       0.84      0.98      0.91       390

    accuracy                           0.88       624
   macro avg       0.90      0.84      0.86       624
weighted avg       0.89      0.88      0.87       624

Transfer Learning Report:
              precision    recall  f1-score   support

           0       0.92      0.76      0.83       234
           1       0.87      0.96      0.91       390

    accuracy                           0.88       624
   macro avg       0.89      0.86      0.87       624
weighted avg       0.89      0.88      0.88       624



In [9]:
from tensorflow.keras.preprocessing import image

def predict_image(img_path, model):
    img = image.load_img(img_path, target_size=(224,224))
    img_array = image.img_to_array(img) / 255.0
    img_array = np.expand_dims(img_array, axis=0)

    pred = model.predict(img_array)[0][0]

    if pred > 0.5:
        return "Pneumonia"
    else:
        return "Normal"

In [10]:
cnn_model.save("cnn_model.keras")
tl_model.save("transfer_model.keras")

In [11]:
tl_model.save("pneumonia_model.keras")

In [12]:
import tensorflow as tf

model = tf.keras.models.load_model("transfer_model.keras")

converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()

with open("model.tflite", "wb") as f:
    f.write(tflite_model)

INFO:tensorflow:Assets written to: C:\Users\gurra\AppData\Local\Temp\tmpwy3eo6m1\assets


INFO:tensorflow:Assets written to: C:\Users\gurra\AppData\Local\Temp\tmpwy3eo6m1\assets


Saved artifact at 'C:\Users\gurra\AppData\Local\Temp\tmpwy3eo6m1'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32, name='input_layer_2')
Output Type:
  TensorSpec(shape=(None, 1), dtype=tf.float32, name=None)
Captures:
  2138057474512: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2138075975120: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2138075980112: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2138075980688: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2138075975696: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2138075979920: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2138075980496: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2138075978960: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2138075974736: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2138075975312: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2138

In [13]:
print(path)

C:\Users\gurra\.cache\kagglehub\datasets\paultimothymooney\chest-xray-pneumonia\versions\2
